# 🧠 Unsupervised Learning - Assignment: Session 2

**Topic:** K-Means Clustering Mechanics: Distance Metrics, Iterative Optimization, Centroid Updates, and the Elbow Method

---

### 📋 Assignment Roadmap
1. **Task 1:** 2D Food Delivery Points Plotting, Random Centroid Initialization & Manual Euclidean Distance Assignment
2. **Task 2:** Custom Python Function `assign_clusters(points, centroids)` using Euclidean Distance
3. **Task 3:** Metric Generalization: Adding Manhattan Distance Support with Metric Selection Parameter
4. **Task 4:** Centroid Recalculation Function `update_centroids(points, assignments, k)` & Convergence Testing
5. **Task 5:** Optimal $k$ Selection via the Elbow Method (WCSS / Inertia Curve on Zomato Restaurant Data)

---

## 1️⃣ Task 1: Food Delivery Locations, Manual Plotting & Centroid Assignment

### 📖 Conceptual Definition & Background
- **The K-Means Clustering Algorithm (Lloyd's Algorithm):** An iterative partitioning method that groups $N$ data observations into $k$ non-overlapping clusters. Each cluster is represented by a prototype coordinate vector called the **Centroid** (mean of the cluster).
- **Step 1: Centroid Initialization:** We choose $k$ initial coordinates $\{c_1, c_2, \dots, c_k\}$.
- **Step 2: Cluster Assignment:** Each delivery location point $P_i(x_i, y_i)$ is assigned to its nearest centroid $C_j(x_j, y_j)$ based on the straight-line **Euclidean Distance ($L_2$ norm)**:

$$d_{\text{Euclidean}}(P_i, C_j) = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2}$$

$$\text{Cluster}(P_i) = \arg\min_{j \in \{1, \dots, k\}} d_{\text{Euclidean}}(P_i, C_j)$$

---

### 📍 Delivery Points & Initial Centroids
We are given 5 food delivery locations on a city grid:
- $P_1 = [2, 3]$
- $P_2 = [5, 8]$
- $P_3 = [1, 2]$
- $P_4 = [6, 9]$
- $P_5 = [7, 7]$

We randomly select **$k=2$ initial centroids**:
- **Centroid 1 ($C_1$):** $[2.0, 2.0]$ (Lower-left quadrant / Downtown)
- **Centroid 2 ($C_2$):** $[6.0, 8.0]$ (Upper-right quadrant / Uptown Suburbs)

---

### 📝 Step-by-Step Manual Euclidean Distance Calculation

| Point | Coordinates $(x, y)$ | Distance to $C_1(2, 2)$ | Distance to $C_2(6, 8)$ | Comparison ($d_1$ vs $d_2$) | Assigned Cluster |
| :---: | :---: | :--- | :--- | :---: | :---: |
| **$P_1$** | $[2, 3]$ | $\sqrt{(2-2)^2 + (3-2)^2} = \sqrt{1} = \mathbf{1.00}$ | $\sqrt{(2-6)^2 + (3-8)^2} = \sqrt{16+25} = \sqrt{41} \approx 6.40$ | $1.00 < 6.40$ | **Cluster 0 ($C_1$)** |
| **$P_2$** | $[5, 8]$ | $\sqrt{(5-2)^2 + (8-2)^2} = \sqrt{9+36} = \sqrt{45} \approx 6.71$ | $\sqrt{(5-6)^2 + (8-8)^2} = \sqrt{1+0} = \mathbf{1.00}$ | $6.71 > 1.00$ | **Cluster 1 ($C_2$)** |
| **$P_3$** | $[1, 2]$ | $\sqrt{(1-2)^2 + (2-2)^2} = \sqrt{1+0} = \mathbf{1.00}$ | $\sqrt{(1-6)^2 + (2-8)^2} = \sqrt{25+36} = \sqrt{61} \approx 7.81$ | $1.00 < 7.81$ | **Cluster 0 ($C_1$)** |
| **$P_4$** | $[6, 9]$ | $\sqrt{(6-2)^2 + (9-2)^2} = \sqrt{16+49} = \sqrt{65} \approx 8.06$ | $\sqrt{(6-6)^2 + (9-8)^2} = \sqrt{0+1} = \mathbf{1.00}$ | $8.06 > 1.00$ | **Cluster 1 ($C_2$)** |
| **$P_5$** | $[7, 7]$ | $\sqrt{(7-2)^2 + (7-2)^2} = \sqrt{25+25} = \sqrt{50} \approx 7.07$ | $\sqrt{(7-6)^2 + (7-8)^2} = \sqrt{1+1} = \sqrt{2} \approx \mathbf{1.41}$ | $7.07 > 1.41$ | **Cluster 1 ($C_2$)** |

- **Cluster 0 ($C_1$ Group):** $\{P_1, P_3\}$
- **Cluster 1 ($C_2$ Group):** $\{P_2, P_4, P_5\}$

In [ ]:
# =====================================================================
# TASK 1: Plotting Delivery Points, Centroids & Initial Assignments
# =====================================================================
import matplotlib.pyplot as plt
import numpy as np

# Given delivery locations
points = np.array([[2, 3], [5, 8], [1, 2], [6, 9], [7, 7]])
point_labels = ["P1 [2,3]", "P2 [5,8]", "P3 [1,2]", "P4 [6,9]", "P5 [7,7]"]

# Two randomly initialized centroids
centroids = np.array([[2.0, 2.0], [6.0, 8.0]])
centroid_labels = ["Centroid C1 [2,2]", "Centroid C2 [6,8]"]

# Assignments determined via Euclidean distance calculation
assignments = [0, 1, 0, 1, 1]  # 0 -> C1, 1 -> C2

# Visualization
fig, ax = plt.subplots(figsize=(9, 7))
cluster_colors = {0: "#1f77b4", 1: "#ff7f0e"}  # Blue for C1, Orange for C2

# 1. Plot delivery points with assigned cluster colors
for i, (pt, cluster_idx) in enumerate(zip(points, assignments)):
    ax.scatter(pt[0], pt[1], color=cluster_colors[cluster_idx], s=140, edgecolors="black", zorder=4)
    ax.annotate(point_labels[i], (pt[0], pt[1]), textcoords="offset points", xytext=(8, -4),
                fontweight="bold", fontsize=10)
    
    # Draw dashed lines connecting point to its assigned centroid
    target_c = centroids[cluster_idx]
    ax.plot([pt[0], target_c[0]], [pt[1], target_c[1]], linestyle="--", color=cluster_colors[cluster_idx],
            alpha=0.6, linewidth=1.5, zorder=2)

# 2. Plot initial centroids
for j, c in enumerate(centroids):
    ax.scatter(c[0], c[1], color=cluster_colors[j], marker="X", s=280, edgecolors="black", linewidth=2,
               label=centroid_labels[j], zorder=5)

ax.set_title("Task 1: Food Delivery Locations & Initial Centroid Assignment (Euclidean)",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("City Grid X-Coordinate (km)", fontsize=11, fontweight="bold")
ax.set_ylabel("City Grid Y-Coordinate (km)", fontsize=11, fontweight="bold")
ax.set_xlim(0, 9)
ax.set_ylim(0, 11)
ax.set_xticks(range(0, 10))
ax.set_yticks(range(0, 12))
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(loc="lower right", frameon=True, facecolor="#fdfefe", edgecolor="#cccccc", fontsize=10)

plt.tight_layout()
plt.show()

## 2️⃣ Task 2: Custom Python Function `assign_clusters(points, centroids)`

### 📖 Conceptual Definition
- **The Assignment Operator in K-Means:** The assignment step partitions the dataset into Voronoi cells. For a set of points $\mathcal{X} = \{x_1, x_2, \dots, x_N\}$ and centroids $\mathcal{C} = \{c_0, c_1, \dots, c_{k-1}\}$:

$$a_i = \arg\min_{j \in \{0, \dots, k-1\}} \sqrt{\sum_{d=1}^D (x_{i, d} - c_{j, d})^2}$$

- **Function Design:**
  - `points`: A sequence of 2D coordinates `[[x1, y1], [x2, y2], ...]`
  - `centroids`: A sequence of centroid coordinates `[[cx0, cy0], [cx1, cy1], ...]`
  - `return`: A list of integer cluster indices `[cluster_idx_0, cluster_idx_1, ...]` representing cluster membership for each point.

In [ ]:
# =====================================================================
# TASK 2: Implementation of assign_clusters using Euclidean Distance
# =====================================================================
import math

def euclidean_distance(point_a, point_b):
    """
    Calculates the Euclidean (L2) distance between two D-dimensional points.
    Formula: sqrt( sum( (a_i - b_i)^2 ) )
    """
    squared_diff_sum = sum((a - b) ** 2 for a, b in zip(point_a, point_b))
    return math.sqrt(squared_diff_sum)


def assign_clusters(points, centroids):
    """
    Assigns each point to the nearest centroid using Euclidean distance.
    
    Parameters:
        points (list of list of float): List of N points.
        centroids (list of list of float): List of k centroids.
        
    Returns:
        assignments (list of int): Cluster index (0 to k-1) for each point.
    """
    assignments = []
    for pt in points:
        distances = [euclidean_distance(pt, c) for c in centroids]
        # Find the index of the minimum distance
        nearest_centroid_idx = distances.index(min(distances))
        assignments.append(nearest_centroid_idx)
    return assignments


# Test with the delivery points and centroids from Task 1
delivery_points = [[2, 3], [5, 8], [1, 2], [6, 9], [7, 7]]
initial_centroids = [[2.0, 2.0], [6.0, 8.0]]

computed_assignments = assign_clusters(delivery_points, initial_centroids)

print("=" * 65)
print("TASK 2: ASSIGN_CLUSTERS (EUCLIDEAN DISTANCE) RESULTS")
print("=" * 65)
for i, (pt, cluster_id) in enumerate(zip(delivery_points, computed_assignments)):
    print(f"Point P{i+1} {pt} -> Assigned to Cluster {cluster_id} (Centroid {initial_centroids[cluster_id]})")

print(f"\nFull Cluster Assignment List: {computed_assignments}")

## 3️⃣ Task 3: Adding Manhattan Distance Support with Parameter Selection

### 📖 Conceptual Definition: Euclidean vs. Manhattan Distance

1. **Euclidean Distance ($L_2$ Norm):**
   $$d_{\text{Euclidean}}(p, c) = \sqrt{\sum_{d=1}^D (p_d - c_d)^2}$$
   - Represents "as-the-crow-flies" straight-line spatial distance.
   - Heavily penalizes large discrepancies along any single dimension due to the squaring operation.

2. **Manhattan Distance ($L_1$ Norm / Taxicab Distance):**
   $$d_{\text{Manhattan}}(p, c) = \sum_{d=1}^D |p_d - c_d|$$
   - Represents navigation strictly along orthogonal city grid avenues and streets (e.g., how a food delivery rider or taxi actually moves through street blocks).
   - More robust to extreme outliers because differences are not squared.

---

### 📊 Metric Comparison on Our Delivery Points ($C_1 = [2, 2]$, $C_2 = [6, 8]$):

| Point | Euclidean $d(P, C_1)$ | Euclidean $d(P, C_2)$ | Manhattan $d(P, C_1)$ | Manhattan $d(P, C_2)$ | Euclidean Assignment | Manhattan Assignment |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **$P_1 [2, 3]$** | **1.00** | 6.40 | $|2-2| + |3-2| = \mathbf{1}$ | $|2-6| + |3-8| = 9$ | **Cluster 0** | **Cluster 0** |
| **$P_2 [5, 8]$** | 6.71 | **1.00** | $|5-2| + |8-2| = 9$ | $|5-6| + |8-8| = \mathbf{1}$ | **Cluster 1** | **Cluster 1** |
| **$P_3 [1, 2]$** | **1.00** | 7.81 | $|1-2| + |2-2| = \mathbf{1}$ | $|1-6| + |2-8| = 11$ | **Cluster 0** | **Cluster 0** |
| **$P_4 [6, 9]$** | 8.06 | **1.00** | $|6-2| + |9-2| = 11$ | $|6-6| + |9-8| = \mathbf{1}$ | **Cluster 1** | **Cluster 1** |
| **$P_5 [7, 7]$** | 7.07 | **1.41** | $|7-2| + |7-2| = 10$ | $|7-6| + |7-8| = \mathbf{2}$ | **Cluster 1** | **Cluster 1** |

In [ ]:
# =====================================================================
# TASK 3: assign_clusters with 'euclidean' and 'manhattan' Options
# =====================================================================
import math
import pandas as pd

def calculate_distance(point_a, point_b, metric='euclidean'):
    """
    Calculates the distance between two points using the specified metric.
    
    Parameters:
        point_a, point_b (iterable): D-dimensional coordinates.
        metric (str): 'euclidean' (L2) or 'manhattan' (L1).
    """
    metric = metric.lower().strip()
    if metric == 'euclidean':
        return math.sqrt(sum((a - b) ** 2 for a, b in zip(point_a, point_b)))
    elif metric == 'manhattan':
        return sum(abs(a - b) for a, b in zip(point_a, point_b))
    else:
        raise ValueError(f"Unsupported distance metric '{metric}'. Choose 'euclidean' or 'manhattan'.")


def assign_clusters(points, centroids, metric='euclidean'):
    """
    Assigns each point to the nearest centroid using the specified distance metric.
    
    Parameters:
        points (list of lists): N points to cluster.
        centroids (list of lists): k cluster centroids.
        metric (str): 'euclidean' (default) or 'manhattan'.
        
    Returns:
        assignments (list of int): Assigned cluster index for each point.
    """
    assignments = []
    for pt in points:
        distances = [calculate_distance(pt, c, metric=metric) for c in centroids]
        nearest_idx = distances.index(min(distances))
        assignments.append(nearest_idx)
    return assignments


# Test both metrics side-by-side
delivery_points = [[2, 3], [5, 8], [1, 2], [6, 9], [7, 7]]
test_centroids = [[2.0, 2.0], [6.0, 8.0]]

euclidean_res = assign_clusters(delivery_points, test_centroids, metric='euclidean')
manhattan_res = assign_clusters(delivery_points, test_centroids, metric='manhattan')

df_comparison = pd.DataFrame({
    "Point": [f"P{i+1}" for i in range(len(delivery_points))],
    "Coordinates": [str(pt) for pt in delivery_points],
    "Euclidean Assignment": [f"Cluster {c}" for c in euclidean_res],
    "Manhattan Assignment": [f"Cluster {c}" for c in manhattan_res],
    "Matches": [e == m for e, m in zip(euclidean_res, manhattan_res)]
})

print("=" * 75)
print("TASK 3: COMPARISON OF EUCLIDEAN VS. MANHATTAN ASSIGNMENTS")
print("=" * 75)
print(df_comparison.to_string(index=False))

# Test error handling with invalid metric
try:
    assign_clusters(delivery_points, test_centroids, metric='cosine')
except ValueError as e:
    print(f"\nInput Validation Test Passed: {e}")

## 4️⃣ Task 4: Centroid Recalculation Function `update_centroids(points, assignments, k)`

### 📖 Conceptual Definition: The Update (M) Step of K-Means
- Once the assignment phase partitions all observations into $k$ subsets $\mathcal{S}_0, \mathcal{S}_1, \dots, \mathcal{S}_{k-1}$, K-Means minimizes the within-cluster variance by recomputing each centroid as the **center of mass (arithmetic mean)** of its constituent points:

$$c_j = \frac{1}{|\mathcal{S}_j|} \sum_{x_i \in \mathcal{S}_j} x_i$$

- **Coordinate-wise Formula in 2D:**
$$c_{j, x} = \frac{1}{|\mathcal{S}_j|} \sum_{x_i \in \mathcal{S}_j} x_{i, 1}, \quad c_{j, y} = \frac{1}{|\mathcal{S}_j|} \sum_{x_i \in \mathcal{S}_j} x_{i, 2}$$

- **Edge Case Handling (Empty Cluster):** If a cluster has no assigned points ($|\mathcal{S}_j| = 0$), division by zero must be prevented by retaining the existing centroid coordinate or reinitializing it.

In [ ]:
# =====================================================================
# TASK 4: Implementation of update_centroids and Convergence Loop
# =====================================================================
import numpy as np

def update_centroids(points, assignments, k):
    """
    Calculates the new centroid coordinates for each cluster as the mean of its assigned points.
    
    Parameters:
        points (list of lists): List of N data points.
        assignments (list of int): Assigned cluster index for each point.
        k (int): Total number of clusters.
        
    Returns:
        new_centroids (list of lists): Recomputed coordinates for each cluster centroid.
    """
    num_features = len(points[0])
    new_centroids = []
    
    for cluster_id in range(k):
        # Filter points belonging to current cluster_id
        cluster_points = [pt for pt, assigned_id in zip(points, assignments) if assigned_id == cluster_id]
        
        if len(cluster_points) == 0:
            # Handle empty cluster safeguard: fallback to origin or keep placeholder
            print(f"Warning: Cluster {cluster_id} has 0 assigned points!")
            new_centroids.append([0.0] * num_features)
        else:
            # Calculate arithmetic mean across each dimension
            centroid_coord = [
                round(sum(pt[dim] for pt in cluster_points) / len(cluster_points), 3)
                for dim in range(num_features)
            ]
            new_centroids.append(centroid_coord)
            
    return new_centroids


# Test on our delivery points dataset
delivery_points = [[2, 3], [5, 8], [1, 2], [6, 9], [7, 7]]
assignments_round1 = [0, 1, 0, 1, 1]
k = 2

new_centroids_round1 = update_centroids(delivery_points, assignments_round1, k)

print("=" * 75)
print("TASK 4: CENTROID RE-ESTIMATION TEST")
print("=" * 75)
print("Initial Centroids : C1 = [2.0, 2.0],  C2 = [6.0, 8.0]")
print(f"Updated Centroids : C1 = {new_centroids_round1[0]}, C2 = {new_centroids_round1[1]}")
print("\nCalculation Verification:")
print(f"  C1 (Points P1[2,3], P3[1,2]): x = (2+1)/2 = 1.5, y = (3+2)/2 = 2.5 -> {new_centroids_round1[0]}")
print(f"  C2 (Points P2[5,8], P4[6,9], P5[7,7]): x = (5+6+7)/3 = 6.0, y = (8+9+7)/3 = 8.0 -> {new_centroids_round1[1]}")

# Demonstration of a Complete 3-Iteration K-Means Optimization Loop
print("\n" + "=" * 75)
print("COMPLETE K-MEANS ITERATIVE OPTIMIZATION WALKTHROUGH:")
print("=" * 75)
curr_centroids = [[2.0, 2.0], [6.0, 8.0]]

for iteration in range(1, 4):
    assigned = assign_clusters(delivery_points, curr_centroids, metric='euclidean')
    next_centroids = update_centroids(delivery_points, assigned, k=2)
    print(f"Iteration {iteration}: Centroids = {curr_centroids} -> Assignments = {assigned} -> Next Centroids = {next_centroids}")
    if next_centroids == curr_centroids:
        print(f"\nCONVERGENCE ACHIEVED at iteration {iteration}! Centroids are perfectly stable.")
        break
    curr_centroids = next_centroids

## 5️⃣ Task 5: The Elbow Method for Optimal $k$ Selection

### 📖 Conceptual Definition: Within-Cluster Sum of Squares (WCSS / Inertia)
- How do we decide the optimal number of clusters $k$ when data has no labels?
- We evaluate **Within-Cluster Sum of Squares (WCSS)**, also termed **Inertia** in `scikit-learn`:

$$\text{WCSS}(k) = \sum_{j=1}^k \sum_{x_i \in \mathcal{C}_j} \|x_i - \mu_j\|^2$$

  Where $\mu_j$ is the centroid of cluster $\mathcal{C}_j$.

### 📐 The "Elbow" Principle
- As $k$ increases, WCSS **always decreases monotonically** (if $k = N$, every point is its own centroid and $\text{WCSS} = 0$).
- However, after a certain $k$, adding more clusters yields only marginal reductions in WCSS.
- The **Elbow Point** represents the optimal tradeoff between model compactness (low variance) and simplicity (avoiding overfitting/redundant clusters).

---

### 🍽️ Dataset Selected: Zomato Restaurant Segmentation
We examine a dataset of **120 restaurants** across two key consumer dining dimensions:
1. **Average Cost for Two (₹):** Price tier (from budget street-eats ₹150 to luxury dining ₹3,000).
2. **Aggregate Customer Rating (1.0 to 5.0):** Quality and satisfaction level.

We test $k \in [1, 6]$ to pinpoint the mathematical elbow.

In [ ]:
# =====================================================================
# TASK 5: Elbow Method using scikit-learn KMeans & Matplotlib
# =====================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Synthesize a realistic Zomato restaurant dining dataset (120 restaurants)
np.random.seed(42)

# Cohort A: Budget Street-food / Fast Casual (Low Cost, High/Decent Rating)
cost_a = np.random.normal(loc=300, scale=80, size=40)
rating_a = np.random.normal(loc=3.8, scale=0.35, size=40)

# Cohort B: Mid-Tier Casual Dining / Family Restos (Moderate Cost, Moderate Rating)
cost_b = np.random.normal(loc=950, scale=150, size=40)
rating_b = np.random.normal(loc=4.1, scale=0.25, size=40)

# Cohort C: Fine Dining / Luxury Lounges (High Cost, Elite Rating)
cost_c = np.random.normal(loc=2200, scale=250, size=40)
rating_c = np.random.normal(loc=4.6, scale=0.20, size=40)

restaurant_df = pd.DataFrame({
    "avg_cost_for_two": np.clip(np.concatenate([cost_a, cost_b, cost_c]), 150, 3000),
    "aggregate_rating": np.clip(np.concatenate([rating_a, rating_b, rating_c]), 1.0, 5.0)
})

# 2. Standardize features to equalize scales (Cost in thousands vs. Rating in single digits)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(restaurant_df)

# 3. Compute WCSS (Inertia) for k = 1 to k = 6
k_values = list(range(1, 7))
wcss_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss_scores.append(km.inertia_)

# 4. Display WCSS Table and Marginal Reductions
wcss_df = pd.DataFrame({
    "Number of Clusters (k)": k_values,
    "WCSS (Inertia)": [round(w, 2) for w in wcss_scores],
    "WCSS Reduction (Delta)": ["-"] + [round(wcss_scores[i-1] - wcss_scores[i], 2) for i in range(1, len(wcss_scores))]
})
print("=" * 65)
print("WCSS / INERTIA TABLE (k=1 to k=6)")
print("=" * 65)
print(wcss_df.to_string(index=False))

# 5. Plot the Elbow Curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: The Elbow Curve
ax1.plot(k_values, wcss_scores, marker='o', markersize=9, linewidth=2.5, color='#2c3e50', label='WCSS Curve')
ax1.scatter([3], [wcss_scores[2]], color='#e74c3c', s=220, zorder=5, label='Optimal Elbow Point (k=3)')
ax1.annotate(
    "Elbow Point (k=3)\nSharp change in curvature",
    xy=(3, wcss_scores[2]),
    xytext=(3.6, wcss_scores[2] + 45),
    arrowprops=dict(facecolor='#e74c3c', shrink=0.08, width=2, headwidth=8),
    fontsize=11,
    fontweight='bold',
    color='#c0392b'
)
ax1.set_title("The Elbow Method: WCSS vs. Number of Clusters (k)", fontsize=13, fontweight='bold', pad=10)
ax1.set_xlabel("Number of Clusters (k)", fontsize=11, fontweight='bold')
ax1.set_ylabel("Within-Cluster Sum of Squares (WCSS / Inertia)", fontsize=11, fontweight='bold')
ax1.set_xticks(k_values)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(frameon=True, facecolor='#f8f9fa', fontsize=10)

# Subplot 2: Resulting Clusters at Optimal k=3
optimal_km = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=10)
restaurant_df["Cluster"] = optimal_km.fit_predict(X_scaled)

cluster_names = {
    0: "Budget Street Eats",
    1: "Mid-Tier Casual Dining",
    2: "Premium Fine Dining"
}
colors_map = ["#27ae60", "#f39c12", "#8e44ad"]

for c_id in range(3):
    subset = restaurant_df[restaurant_df["Cluster"] == c_id]
    ax2.scatter(subset["avg_cost_for_two"], subset["aggregate_rating"],
                label=f"Cluster {c_id}: {cluster_names[c_id]}", color=colors_map[c_id],
                s=70, alpha=0.85, edgecolors='black', linewidth=0.5)

ax2.set_title("Zomato Restaurant Clusters at Optimal k=3", fontsize=13, fontweight='bold', pad=10)
ax2.set_xlabel("Average Cost for Two (₹)", fontsize=11, fontweight='bold')
ax2.set_ylabel("Aggregate Rating (1 to 5 Stars)", fontsize=11, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(frameon=True, facecolor='#f8f9fa', fontsize=10)

plt.tight_layout()
plt.show()

### 🔍 Detailed Analysis of the Elbow Point ($k=3$):
1. **Steep Slope ($k=1 \rightarrow k=2 \rightarrow k=3$):** Transitioning from $k=1$ to $k=2$ and then to $k=3$ drops the WCSS drastically (a huge reduction in variance), capturing the broad natural categories of restaurants (*Budget*, *Mid-Tier Casual*, *Fine Dining*).
2. **Plateau ($k \ge 4$):** Beyond $k=3$, the rate of decrease in WCSS flattens out noticeably. Splitting the data into 4 or more clusters merely subdivides existing natural cohorts without uncovering significant new structural boundaries.
3. **Optimal Selection:** Hence, $k=3$ represents the true mathematical and business **elbow point**.

---

## 🏁 Session 2 Summary & Takeaways
1. **Core Iteration Loop:** K-Means operates through two alternating steps:
   - **Expectation / Assignment Step:** Assigns points to the closest centroid ($L_1$ Manhattan or $L_2$ Euclidean).
   - **Maximization / Update Step:** Recomputes centroids as the arithmetic mean of all assigned points.
2. **Distance Metric Choice:** Euclidean ($L_2$) captures direct geometric proximity, whereas Manhattan ($L_1$) aligns with orthogonal grid travel (e.g., street navigation) and is more outlier-resilient.
3. **Convergence Guarantee:** Lloyd's algorithm is guaranteed to converge because the objective function (WCSS) is monotonically non-increasing at each step.
4. **Hyperparameter Tuning:** The Elbow method (WCSS vs $k$) provides an objective mathematical criterion for selecting $k$ without ground-truth labels.